<h1><p style="text-align: center;">Quickstart for MacTrack2</p></h1>

Through this file, you can find all the information you need to make this project your own. Don't forget to create your Python environment with the packages available in the `environment.txt` file.

> WARNING! When you need to make actions, the details will be preceded by this style.

### Table of contents

* [How to create your own model](#chapter1)  
  * [Dataset](#dataset)
  * [Training](#training)
  * [Testing](#testing)

## How to create your own model <a class="anchor" id="chapter1"></a>

### Dataset <a class="anchor" id="dataset"></a>

First, you need a dataset to create your model and a *models* folder to store it. 

In [ ]:
import os

dirs = [
    "model",
    "model/dataset",
    "model/dataset/train",
    "model/dataset/train/train_x",
    "model/dataset/train/train_y",
    "model/dataset/test",
    "model/dataset/test/test_x",
    "model/dataset/test/test_y",
    "model/models",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

> This dataset is essential to create your model, you need to add the following :  
> - `train_x` : a training set of images that you hand-cut yourself using [Fiji](https://imagej.net/software/fiji/downloads)
> - `train_y` : the masks for each segmented frame in a zip file containing all the ROI files
> - `test_x` and `test_y` : you should do the same, it will serve to test the model that has been trained on the training set.

Here is an example of an image that will be placed in the *x* sets (wether it is the training or the test set) on the left. And on the right, each yellow lining represent a ROI (region of interest). Those are saved in a zip file and in the *y* sets.  
<p align="center">
  <img src="images/example_frame.jpg" alt="frame_ex" width="45%"/>
  <img src="images/example_frame_ROI.jpg" alt="ROI_ex" width="45%"/>
</p>


(TODO : DOC ABOUT FIJI)

Next, the `kartezio` package that will help us build the model need two other files : `META.json` and `dataset.csv`.

The META file is here to help the package recognize the formats of the different objects in the dataset folder.  
> You need to run the following to help you create it.

In [ ]:
import os
import json

# Path to the JSON file
meta_path = "model/dataset/META.json"

# Content of the JSON file
meta_data = {
    "name": "Macrophage",
    "scale": 1.0,
    "label_name": "macrophage",
    "mode": "dataframe",
    "input": {
        "type": "image",
        "format": "hsv"
    },
    "label": {
        "type": "roi",
        "format": "polygon"
    }
}

# Create the JSON file with the content in it
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta_data, f, indent=4)

print(f"Fichier {meta_path} créé avec succès !")

Fichier model2/dataset/META.json créé avec succès !


The csv file is here to help kartezio read your dataset.

In [ ]:
from Set_up import dataset_csv
dataset_csv.create_dataset_csv(input_folder='model/dataset', output_csv='model/dataset/dataset.csv')

Your two files are now created and located in the `model/dataset` folder. We can now train our model.

### Training <a class="anchor" id="training"></a>

Now that we have the structure kartezio needs to function correctly, we can create and train a model using the `create_segmentation_model` function of the kartezio package. You can find the following code in the `Set_up/train_model.py` file.

In [4]:
from kartezio.apps.segmentation import create_segmentation_model
from kartezio.endpoint import EndpointThreshold
from kartezio.dataset import read_dataset
from kartezio.training import train_model

DATASET = "model2/dataset"
OUTPUT = "model2/models"

generations = 1000
_lambda = 5
frequency = 5
rate = 0.1
print(rate)
model = create_segmentation_model(
    generations,
    _lambda,
    inputs=3,
    nodes=30,
    node_mutation_rate=rate,
    output_mutation_rate=rate,
    outputs=1,
    fitness="IOU",
    endpoint=EndpointThreshold(threshold=4)
)

dataset = read_dataset(DATASET)
elite, a = train_model(model, dataset, OUTPUT, callback_frequency=frequency)

[Kartezio - INFO] -  42 nodes registered.
[Kartezio - INFO] -  5 metrics registered.
[Kartezio - INFO] -  7 fitness registered.
[Kartezio - INFO] -  7 endpoints registered.
[Kartezio - INFO] -  5 stackers registered.
0.1
Files will be saved under model2/models/301185-4bc6e300-73e2-49ab-b905-261f2f33f486.
[G 0005] 0.6324 0.006600s 152fps
[G 0010] 0.6324 0.004849s 206fps


/home/gbouland/micromamba/envs/sam-env/lib/python3.12/site-packages/kartezio/image/nodes.py:238: RuntimeWarning: invalid value encountered in cast
  return cv2.Sobel(x[0], cv2.CV_64F, 1, 0, ksize=ksize).astype(np.uint8)


[G 0015] 0.6021 0.007831s 128fps
[G 0020] 0.6021 0.007997s 125fps
[G 0025] 0.5573 0.016767s 60fps
[G 0030] 0.4846 0.035180s 28fps


/home/gbouland/micromamba/envs/sam-env/lib/python3.12/site-packages/kartezio/image/nodes.py:239: RuntimeWarning: invalid value encountered in cast
  return cv2.Sobel(x[0], cv2.CV_64F, 0, 1, ksize=ksize).astype(np.uint8)


[G 0035] 0.4846 0.035403s 28fps
[G 0040] 0.4846 0.035262s 28fps
[G 0045] 0.4846 0.035021s 29fps
[G 0050] 0.4846 0.035858s 28fps
[G 0055] 0.4846 0.035192s 28fps
[G 0060] 0.4846 0.034968s 29fps
[G 0065] 0.4846 0.035605s 28fps
[G 0070] 0.4846 0.035358s 28fps
[G 0075] 0.4846 0.035255s 28fps
[G 0080] 0.4846 0.036006s 28fps
[G 0085] 0.4846 0.036003s 28fps
[G 0090] 0.3499 0.038198s 26fps
[G 0095] 0.3499 0.037611s 27fps
[G 0100] 0.3499 0.037440s 27fps
[G 0105] 0.3499 0.038113s 26fps
[G 0110] 0.3499 0.037001s 27fps
[G 0115] 0.3499 0.037095s 27fps
[G 0120] 0.3499 0.037775s 26fps
[G 0125] 0.3499 0.037471s 27fps
[G 0130] 0.3499 0.040025s 25fps
[G 0135] 0.3499 0.037585s 27fps
[G 0140] 0.3499 0.037641s 27fps
[G 0145] 0.3499 0.037123s 27fps
[G 0150] 0.3499 0.039307s 25fps
[G 0155] 0.3499 0.038118s 26fps
[G 0160] 0.3499 0.037219s 27fps
[G 0165] 0.3499 0.037660s 27fps
[G 0170] 0.3499 0.038181s 26fps
[G 0175] 0.3499 0.037621s 27fps
[G 0180] 0.3499 0.037489s 27fps
[G 0185] 0.3499 0.036976s 27fps
[G 0190]

/home/gbouland/micromamba/envs/sam-env/lib/python3.12/site-packages/kartezio/image/nodes.py:322: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  kur = np.mean(kurtosis(img, fisher=True))
/home/gbouland/micromamba/envs/sam-env/lib/python3.12/site-packages/kartezio/image/nodes.py:323: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  skew1 = np.mean(skew(img))


[G 0890] 0.3473 0.027247s 37fps
[G 0895] 0.3473 0.026028s 38fps
[G 0900] 0.3473 0.026545s 38fps
[G 0905] 0.3473 0.026659s 38fps
[G 0910] 0.3473 0.027063s 37fps
[G 0915] 0.3473 0.026308s 38fps
[G 0920] 0.3473 0.026341s 38fps
[G 0925] 0.3473 0.026439s 38fps
[G 0930] 0.3473 0.026162s 38fps
[G 0935] 0.3473 0.026906s 37fps
[G 0940] 0.3473 0.026394s 38fps
[G 0945] 0.3473 0.026321s 38fps
[G 0950] 0.3473 0.027875s 36fps
[G 0955] 0.3473 0.027741s 36fps
[G 0960] 0.3473 0.028008s 36fps
[G 0965] 0.3473 0.026880s 37fps
[G 0970] 0.3473 0.026944s 37fps
[G 0975] 0.3473 0.026414s 38fps
[G 0980] 0.3473 0.026820s 37fps
[G 0985] 0.3473 0.026908s 37fps
[G 0990] 0.3473 0.026538s 38fps
[G 0995] 0.3473 0.026946s 37fps
[G 1000] 0.3473 0.027431s 36fps
[G 1000] 0.3473 0.027431s 36fps, loop done.
All generations packed in model2/models/301185-4bc6e300-73e2-49ab-b905-261f2f33f486.
All 200 generation files deleted.


### Testing <a class="anchor" id="testing"></a>

Now that you trained your model, you can test it to see if it gives good prediction.

In [5]:
import numpy as np
import pandas as pd

from kartezio.easy import print_stats
from kartezio.dataset import read_dataset
from kartezio.fitness import FitnessIOU
from kartezio.inference import ModelPool

scores_all = {}
pool = ModelPool(f"model2/models", FitnessIOU(), regex="*/elite.json").to_ensemble()
dataset = read_dataset(f"model2/dataset", counting=True)
annotations_test = 0
annotations_training = 0
roi_pixel_areas = []
for y_true in dataset.train_y:
    n_annotations = y_true[1]
    annotations_training += n_annotations
for y_true in dataset.test_y:
    annotations = y_true[0]
    n_annotations = y_true[1]
    annotations_test += n_annotations
    for i in range(1, n_annotations + 1):
        roi_pixel_areas.append(np.count_nonzero(annotations[annotations == i]))
print(f"Total annotations for training set: {annotations_training}")
print(f"Total annotations for test set: {annotations_test}")
print(f"Mean pixel area for test set: {np.mean(roi_pixel_areas)}")


scores_test = []
scores_training = []
for i, model in enumerate(pool.models):
    # Test set
    _, fitness, _ = model.eval(dataset, subset="test")
    scores_test.append(1.0 - fitness)

    # Training set
    _, fitness, _ = model.eval(dataset, subset="train")
    scores_training.append(1.0 - fitness)


scores_all[f"training"] = scores_training
scores_all[f"test"] = scores_test
print_stats(scores_training, "IOU", "training set")
print_stats(scores_test, "IOU", "test set")

pd.DataFrame(scores_all).to_csv("model2/results.csv", index=False)

Total annotations for training set: 291
Total annotations for test set: 66
Mean pixel area for test set: 2633.712121212121
-- Statistics for training set, using IOU fitness:
Min 	 Mean +/- SD 	 Max
0.653 	 0.653+/-0.000 	 0.653
-- Statistics for test set, using IOU fitness:
Min 	 Mean +/- SD 	 Max
0.531 	 0.531+/-0.000 	 0.531
